<a href="https://colab.research.google.com/github/icosrle31/Test_ENNOH/blob/main/Demand_profiles_reading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction
packages installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
# Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


ROOT_DIR = os.getcwd()
PROJECT_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/data")


In [ ]:
zones = ["ITSI", "ITSA", "ITS1", "ITN1", "ITCS", "ITCN", "ITCA"]

# Demand Data - GA and DE

In [ ]:
year = 2040 # 2040 or 2050
scenario = "GA" # GA or DE

## Hydrogen Demand Zone 1 and Zone 2

In [ ]:
import os
import pandas as pd

# year, scenario, and zones are already defined in the notebook

# Assuming the base demand profile folder path
demand_base_path = os.path.join(PROJECT_DIR, 'Demand_profiles')

# Construct the path based on scenario and year (no country needed)
scenario_year_path = os.path.join(demand_base_path, scenario, str(year))


In [ ]:
scenario_year_path

'/content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/data/Demand_profiles/GA/2040'

In [ ]:
import pandas as pd
import numpy as np
import os
import calendar

# Determine if it's a leap year
is_leap = calendar.isleap(year)
total_hours = 8784 if is_leap else 8760

# Create the time index dynamically based on the 'year' variable and leap year status
time_index = pd.date_range(start=f"{year}-01-01 00:00", periods=total_hours, freq="h")

def process_and_extrapolate(df):
    # Extract the profile values and ensure it's exactly 8760 long initially
    raw_profile = df.values.flatten()

    profile_8760 = np.zeros(8760)
    copy_len = min(len(raw_profile), 8760)
    profile_8760[:copy_len] = raw_profile[:copy_len]

    if is_leap:
        # Extrapolate for Feb 29 (Leap year: duplicate Feb 28th)
        feb_29_start = 59 * 24
        feb_28_start = 58 * 24
        feb_28_data = profile_8760[feb_28_start : feb_28_start + 24]

        profile_final = np.zeros(8784)
        profile_final[:feb_29_start] = profile_8760[:feb_29_start]
        profile_final[feb_29_start : feb_29_start + 24] = feb_28_data
        profile_final[feb_29_start + 24 :] = profile_8760[feb_28_start + 24 :]
    else:
        profile_final = profile_8760

    # Return a Series with the correct time index
    return pd.Series(profile_final, index=time_index)

def load_zonal_profiles(file_path, zones):
    print(f"Loading {os.path.basename(file_path)}...")

    # Debug: Print available sheet names to figure out mapping
    try:
        xls = pd.ExcelFile(file_path)
        print(f"  -> AVAILABLE SHEETS IN FILE: {xls.sheet_names}")
    except Exception as e:
        print(f"  -> Could not read sheet names: {e}")

    combined_df = pd.DataFrame(index=time_index)
    for zone in zones:
        try:
            # Read specific sheet for the zone, skip 10 rows, use column AD
            df_temp = pd.read_excel(file_path, sheet_name=zone, skiprows=10, usecols="AD")
            # Extrapolate and assign to the zone column
            combined_df[zone] = process_and_extrapolate(df_temp)
            print(f"  - Successfully loaded zone: {zone}")
        except Exception as e:
            print(f"  - Error loading zone {zone}: {e}. Filling with zeros.")
            combined_df[zone] = np.zeros(total_hours)
    return combined_df

def load_hydrogen_profiles(scenario, year, zones):
    """
    Loads Hydrogen profiles for Zone 1 and Zone 2 based on the scenario and year.
    """
    print(f"--- Loading Hydrogen Profiles ({scenario} {year}) ---")

    # Construct paths
    demand_base_path = os.path.join(PROJECT_DIR, 'Demand_profiles')
    scenario_year_path = os.path.join(demand_base_path, scenario, str(year))

    h2_zone_1_path = os.path.join(scenario_year_path, 'H2_ZONE_1.xlsx')
    h2_zone_2_path = os.path.join(scenario_year_path, 'H2_ZONE_2.xlsx')

    # Load profiles
    h2_zone_1_df = load_zonal_profiles(h2_zone_1_path, zones)
    h2_zone_2_df = load_zonal_profiles(h2_zone_2_path, zones)

    print(f"\nSuccessfully loaded H2_ZONE_1 (Shape: {h2_zone_1_df.shape})")
    print(f"Successfully loaded H2_ZONE_2 (Shape: {h2_zone_2_df.shape})")

    return h2_zone_1_df, h2_zone_2_df

# Example usage:
h2_zone_1, h2_zone_2 = load_hydrogen_profiles(scenario, year, zones)
display(h2_zone_1.head())
display(h2_zone_2.head())

--- Loading Hydrogen Profiles (GA 2040) ---
Loading H2_ZONE_1.xlsx...
  -> AVAILABLE SHEETS IN FILE: ['AT00', 'BE00', 'BG00', 'CH00', 'CY00', 'CZ00', 'DE00', 'DKE1', 'EE00', 'ES00', 'FI00', 'FR00', 'GR00', 'HU00', 'HR00', 'IE00', 'ITN1', 'LUG1', 'LT00', 'LV00', 'MT00', 'NL00', 'NO00', 'PL00', 'PT00', 'RO00', 'SK00', 'SI00', 'SE00', 'UK00']
  - Error loading zone ITSI: Worksheet named 'ITSI' not found. Filling with zeros.
  - Error loading zone ITSA: Worksheet named 'ITSA' not found. Filling with zeros.
  - Error loading zone ITS1: Worksheet named 'ITS1' not found. Filling with zeros.
  - Successfully loaded zone: ITN1
  - Error loading zone ITCS: Worksheet named 'ITCS' not found. Filling with zeros.
  - Error loading zone ITCN: Worksheet named 'ITCN' not found. Filling with zeros.
  - Error loading zone ITCA: Worksheet named 'ITCA' not found. Filling with zeros.
Loading H2_ZONE_2.xlsx...
  -> AVAILABLE SHEETS IN FILE: ['AT00', 'BE00', 'BG00', 'CH00', 'CY00', 'CZ00', 'DE00', 'DKE1', 'EE

,ITSI,ITSA,ITS1,ITN1,ITCS,ITCN,ITCA
2040-01-01 00:00:00,0.0,0.0,0.0,2109.913047,0.0,0.0,0.0
2040-01-01 01:00:00,0.0,0.0,0.0,2109.982283,0.0,0.0,0.0
2040-01-01 02:00:00,0.0,0.0,0.0,2110.051518,0.0,0.0,0.0
2040-01-01 03:00:00,0.0,0.0,0.0,2110.120754,0.0,0.0,0.0
2040-01-01 04:00:00,0.0,0.0,0.0,2110.189990,0.0,0.0,0.0


,ITSI,ITSA,ITS1,ITN1,ITCS,ITCN,ITCA
2040-01-01 00:00:00,0.0,0.0,0.0,9638.589755,0.0,0.0,0.0
2040-01-01 01:00:00,0.0,0.0,0.0,9746.624120,0.0,0.0,0.0
2040-01-01 02:00:00,0.0,0.0,0.0,10124.734289,0.0,0.0,0.0
2040-01-01 03:00:00,0.0,0.0,0.0,10961.999615,0.0,0.0,0.0
2040-01-01 04:00:00,0.0,0.0,0.0,12957.146807,0.0,0.0,0.0


## Heat Demand - CH4 (methane) and H2 (hydrogen)

In [ ]:
import pandas as pd
import os

def load_heat_profiles(scenario, year, zones):
    """
    Loads H2 and CH4 Heat Demand profiles based on scenario and year.
    """
    print(f"--- Loading Heat Profiles ({scenario} {year}) ---")

    demand_base_path = os.path.join(PROJECT_DIR, 'Demand_profiles')
    scenario_year_path = os.path.join(demand_base_path, scenario, str(year))

    # Construct the file paths dynamically
    h2_heat_demand_path = os.path.join(scenario_year_path, f'H2 HEAT DEMAND {scenario} {year}.xlsx')
    ch4_heat_demand_path = os.path.join(scenario_year_path, f'CH4 HEAT DEMAND {scenario} {year}.xlsx')

    # Reload profiles to apply updated time index
    h2_heat_df = load_zonal_profiles(h2_heat_demand_path, zones)
    print(f"\nSuccessfully loaded H2 HEAT DEMAND {scenario} {year} (Shape: {h2_heat_df.shape})")

    ch4_heat_df = load_zonal_profiles(ch4_heat_demand_path, zones)
    print(f"\nSuccessfully loaded CH4 HEAT DEMAND {scenario} {year} (Shape: {ch4_heat_df.shape})")

    return h2_heat_df, ch4_heat_df

# Example usage
h2_heat_df, ch4_heat_df = load_heat_profiles(scenario, year, zones)
display(h2_heat_df.head())
display(ch4_heat_df.head())


--- Loading Heat Profiles (GA 2040) ---
Loading H2 HEAT DEMAND GA 2040.xlsx...
  -> AVAILABLE SHEETS IN FILE: ['AT00', 'BE00', 'BG00', 'CZ00', 'CY00', 'DE00', 'DKE1', 'DKW1', 'EE00', 'ES00', 'FI00', 'FR00', 'GR00', 'HU00', 'HR00', 'IE00', 'ITN1', 'ITS1', 'ITCN', 'ITCS', 'ITSI', 'ITSA', 'ITCA', 'LUG1', 'LT00', 'LV00', 'MT00', 'NL00', 'PL00', 'PT00', 'RO00', 'SK00', 'SI00', 'SE01', 'SE02', 'SE03', 'SE04', 'UKNI', 'UK00']
  - Successfully loaded zone: ITSI
  - Successfully loaded zone: ITSA
  - Successfully loaded zone: ITS1
  - Successfully loaded zone: ITN1
  - Successfully loaded zone: ITCS
  - Successfully loaded zone: ITCN
  - Successfully loaded zone: ITCA

Successfully loaded H2 HEAT DEMAND GA 2040 (Shape: (8784, 7))
Loading CH4 HEAT DEMAND GA 2040.xlsx...
  -> AVAILABLE SHEETS IN FILE: ['AT00', 'BE00', 'BG00', 'HR00', 'CZ00', 'CY00', 'FI00', 'FR00', 'DE00', 'DKE1', 'DKW1', 'GR00', 'HU00', 'ITN1', 'ITS1', 'ITCN', 'ITCS', 'ITSI', 'ITSA', 'ITCA', 'LUG1', 'MT00', 'NL00', 'PL00', 'PT00

,ITSI,ITSA,ITS1,ITN1,ITCS,ITCN,ITCA
2040-01-01 00:00:00,905.285625,363.658227,1063.081098,8456.370791,2170.203418,1306.106674,255.532464
2040-01-01 01:00:00,904.247097,363.136724,1061.358390,8441.880414,2173.298504,1305.371543,255.479090
2040-01-01 02:00:00,994.318173,401.839353,1168.150159,9279.128425,2404.575284,1439.060942,281.584037
2040-01-01 03:00:00,1225.170282,498.007949,1446.901164,11433.526760,2976.678676,1775.858952,347.325056
2040-01-01 04:00:00,1701.112830,697.832434,2009.902000,15840.777727,4149.280273,2463.165521,482.895680


,ITSI,ITSA,ITS1,ITN1,ITCS,ITCN,ITCA
2040-01-01 00:00:00,1013.652765,408.858421,1190.323458,9464.464112,2434.769381,1462.319965,286.424915
2040-01-01 01:00:00,1012.773889,408.429294,1188.734051,9450.853792,2438.957923,1461.893700,286.449450
2040-01-01 02:00:00,1116.139825,453.274023,1311.242955,10410.519344,2705.273095,1615.121810,316.476834
2040-01-01 03:00:00,1379.734844,564.132100,1629.254485,12867.631515,3361.220441,1999.465674,391.735025
2040-01-01 04:00:00,1923.770538,794.729768,2272.635755,17900.573003,4707.597945,2784.881737,547.112117


## Electricity Market and Prosumers

In [ ]:
import pandas as pd
import os

def load_electricity_profiles(scenario, year, zones):
    """
    Loads Electricity Market and Prosumer profiles based on scenario and year.
    """
    print(f"--- Loading Electricity Profiles ({scenario} {year}) ---")

    demand_base_path = os.path.join(PROJECT_DIR, 'Demand_profiles')
    scenario_year_path = os.path.join(demand_base_path, scenario, str(year))

    # Construct the file paths dynamically based on scenario and year
    electricity_market_path = os.path.join(scenario_year_path, f'ELECTRICITY_MARKET {scenario} {year}.xlsx')
    electricity_prosumer_path = os.path.join(scenario_year_path, f'ELECTRICITY_PROSUMER {scenario} {year}.xlsx')

    # Load Zonal Profiles for Electricity Market
    electricity_market_df = load_zonal_profiles(electricity_market_path, zones)
    print(f"\nSuccessfully loaded ELECTRICITY_MARKET {scenario} {year} (Shape: {electricity_market_df.shape})")

    # Load Zonal Profiles for Electricity Prosumer
    electricity_prosumer_df = load_zonal_profiles(electricity_prosumer_path, zones)
    print(f"\nSuccessfully loaded ELECTRICITY_PROSUMER {scenario} {year} (Shape: {electricity_prosumer_df.shape})")

    return electricity_market_df, electricity_prosumer_df

# Example usage
electricity_market_df, electricity_prosumer_df = load_electricity_profiles(scenario, year, zones)
display(electricity_market_df.head())
display(electricity_prosumer_df.head())


--- Loading Electricity Profiles (GA 2040) ---
Loading ELECTRICITY_MARKET GA 2040.xlsx...
  -> AVAILABLE SHEETS IN FILE: ['AL00', 'AT00', 'BA00', 'BE00', 'BG00', 'CH00', 'CY00', 'CZ00', 'DE00', 'DKE1', 'DKW1', 'EE00', 'ES00', 'FI00', 'FR00', 'GR00', 'GR03', 'HR00', 'HU00', 'IE00', 'ITCA', 'ITCN', 'ITCS', 'ITN1', 'ITS1', 'ITSA', 'ITSI', 'LT00', 'LUB1', 'LUF1', 'LUG1', 'LV00', 'ME00', 'MK00', 'MT00', 'NL00', 'NOM1', 'NON1', 'NOS0', 'PL00', 'PT00', 'RO00', 'RS00', 'SE01', 'SE02', 'SE03', 'SE04', 'SI00', 'SK00', 'UK00', 'UKNI']
  - Successfully loaded zone: ITSI
  - Successfully loaded zone: ITSA
  - Successfully loaded zone: ITS1
  - Successfully loaded zone: ITN1
  - Successfully loaded zone: ITCS
  - Successfully loaded zone: ITCN
  - Successfully loaded zone: ITCA

Successfully loaded ELECTRICITY_MARKET GA 2040 (Shape: (8784, 7))
Loading ELECTRICITY_PROSUMER GA 2040.xlsx...
  -> AVAILABLE SHEETS IN FILE: ['AT00', 'BE00', 'BG00', 'CY00', 'CZ00', 'DE00', 'DKE1', 'DKW1', 'EE00', 'ES00', '

,ITSI,ITSA,ITS1,ITN1,ITCS,ITCN,ITCA
2040-01-01 00:00:00,2009.0000,2009.0000,2009.0000,2009.0000,2009.0000,2009.0000,2009.0000
2040-01-01 01:00:00,893.5234,767.4366,1021.3973,5427.4143,2270.2603,956.2861,295.2180
2040-01-01 02:00:00,837.1671,740.1678,959.0427,5149.5444,1956.6239,923.3763,268.8070
2040-01-01 03:00:00,802.7096,718.1430,924.5467,5010.4670,1875.7662,913.4741,257.3263
2040-01-01 04:00:00,775.9906,709.1712,909.5163,5013.8319,1847.2454,880.8261,251.0431


,ITSI,ITSA,ITS1,ITN1,ITCS,ITCN,ITCA
2040-01-01 00:00:00,2009.0000,2009.0000,2009.0000,2009.0000,2009.0000,2009.0000,2009.0000
2040-01-01 01:00:00,1117.9624,267.5718,1625.0157,8732.5441,2840.4257,1368.9562,340.0821
2040-01-01 02:00:00,1080.4440,262.8934,1585.3135,8531.5255,2565.0163,1336.9868,320.3563
2040-01-01 03:00:00,1070.9570,260.1844,1545.1280,8468.9538,2464.7047,1308.9017,310.2249
2040-01-01 04:00:00,1085.4918,261.6034,1505.9628,8486.7642,2437.1029,1271.5444,312.4803


## Synthetic Fuels

In [ ]:
import pandas as pd
import numpy as np
import os

def load_synthetic_fuels(scenario, year):
    """
    Loads Synthetic Fuels profiles (SNG, e-kerosene, e-diesel) based on scenario and year.
    """
    print(f"--- Loading Synthetic Fuels Profiles ({scenario} {year}) ---")

    demand_base_path = os.path.join(PROJECT_DIR, 'Demand_profiles')
    scenario_year_path = os.path.join(demand_base_path, scenario, str(year))
    synth_fuels_path = os.path.join(scenario_year_path, 'SYNTHETIC FUELS.xlsx')

    print(f"Loading {os.path.basename(synth_fuels_path)}...")

    # Debug: Print available sheet names
    try:
        xls = pd.ExcelFile(synth_fuels_path)
        print(f"  -> AVAILABLE SHEETS IN FILE: {xls.sheet_names}")
    except Exception as e:
        print(f"  -> Could not read sheet names: {e}")

    # Mapping the correct sheet names to the specific dataframe columns
    sheets_to_columns = {
        'sng': 'sng',
        'e-kerosene': 'e-kerosine',
        'e-diesel': 'e-diesele'
    }

    # Initialize the dataframe with the dynamic time index
    synthetic_fuels_df = pd.DataFrame(index=time_index)

    for sheet, col_name in sheets_to_columns.items():
        try:
            # Read specific sheet, skip 10 rows, use column AD
            df_temp = pd.read_excel(synth_fuels_path, sheet_name=sheet, skiprows=10, usecols="AD")

            # Extrapolate and assign to the appropriate column
            synthetic_fuels_df[col_name] = process_and_extrapolate(df_temp)
            print(f"  - Successfully loaded sheet '{sheet}' into column '{col_name}'")
        except Exception as e:
            print(f"  - Error loading sheet '{sheet}': {e}. Filling with zeros.")
            synthetic_fuels_df[col_name] = np.zeros(total_hours)

    print(f"\nSuccessfully loaded Synthetic Fuels (Shape: {synthetic_fuels_df.shape})")
    return synthetic_fuels_df

# Example usage
synthetic_fuels_df = load_synthetic_fuels(scenario, year)
display(synthetic_fuels_df.head())


--- Loading Synthetic Fuels Profiles (GA 2040) ---
Loading SYNTHETIC FUELS.xlsx...
  -> AVAILABLE SHEETS IN FILE: ['e-diesel', 'e-kerosene', 'sng']
  - Successfully loaded sheet 'sng' into column 'sng'
  - Successfully loaded sheet 'e-kerosene' into column 'e-kerosine'
  - Successfully loaded sheet 'e-diesel' into column 'e-diesele'

Successfully loaded Synthetic Fuels (Shape: (8784, 3))


,sng,e-kerosine,e-diesele
2040-01-01 00:00:00,13584.474886,15152.332374,17922.374429
2040-01-01 01:00:00,13584.474886,15152.332374,17922.374429
2040-01-01 02:00:00,13584.474886,15152.332374,17922.374429
2040-01-01 03:00:00,13584.474886,15152.332374,17922.374429
2040-01-01 04:00:00,13584.474886,15152.332374,17922.374429


## Saving data

In [ ]:
import pandas as pd

# Create an empty list to collect all individual series
all_series = []

for zone in zones:
    # H2 Zone 1
    if zone in h2_zone_1.columns:
        all_series.append(h2_zone_1[zone].rename(f"{zone}_H2_zone_1"))

    # H2 Zone 2
    if zone in h2_zone_2.columns:
        all_series.append(h2_zone_2[zone].rename(f"{zone}_H2_zone_2"))

    # H2 Heat
    if zone in h2_heat_df.columns:
        all_series.append(h2_heat_df[zone].rename(f"{zone}_H2_heat"))

    # CH4 Heat
    if zone in ch4_heat_df.columns:
        all_series.append(ch4_heat_df[zone].rename(f"{zone}_CH4_heat"))

    # Electricity Market
    if 'electricity_market_df' in globals() and zone in electricity_market_df.columns:
        all_series.append(electricity_market_df[zone].rename(f"{zone}_El_market"))
    elif 'elec_market_nt' in globals() and zone in elec_market_nt.columns:
        # Fallback to NT scenario if applicable
        all_series.append(elec_market_nt[zone].rename(f"{zone}_El_market"))

    # Electricity Prosumer
    if 'electricity_prosumer_df' in globals() and zone in electricity_prosumer_df.columns:
        all_series.append(electricity_prosumer_df[zone].rename(f"{zone}_El_prosumer"))

    # Synthetic Fuels (duplicate for each zone as per prompt)
    if 'sng' in synthetic_fuels_df.columns:
        all_series.append(synthetic_fuels_df['sng'].rename(f"{zone}_SNG"))
    if 'e-diesele' in synthetic_fuels_df.columns:
        all_series.append(synthetic_fuels_df['e-diesele'].rename(f"{zone}_eDiesel"))
    if 'e-kerosine' in synthetic_fuels_df.columns:
        all_series.append(synthetic_fuels_df['e-kerosine'].rename(f"{zone}_eKerosine"))

# Concatenate all series into a single DataFrame
demand_profiles_df = pd.concat(all_series, axis=1)

# Assign to the dynamically named variable
df_name = f"demand_profiles_{scenario}_{year}"
globals()[df_name] = demand_profiles_df

print(f"Created dataframe '{df_name}' with shape: {demand_profiles_df.shape}")
display(demand_profiles_df.head(5))


Created dataframe 'demand_profiles_GA_2040' with shape: (8784, 63)


,ITSI_H2_zone_1,ITSI_H2_zone_2,ITSI_H2_heat,ITSI_CH4_heat,ITSI_El_market,ITSI_El_prosumer,ITSI_SNG,ITSI_eDiesel,ITSI_eKerosine,ITSA_H2_zone_1,...,ITCN_eKerosine,ITCA_H2_zone_1,ITCA_H2_zone_2,ITCA_H2_heat,ITCA_CH4_heat,ITCA_El_market,ITCA_El_prosumer,ITCA_SNG,ITCA_eDiesel,ITCA_eKerosine
2040-01-01 00:00:00,0.0,0.0,905.285625,1013.652765,2009.0000,2009.0000,13584.474886,17922.374429,15152.332374,0.0,...,15152.332374,0.0,0.0,255.532464,286.424915,2009.0000,2009.0000,13584.474886,17922.374429,15152.332374
2040-01-01 01:00:00,0.0,0.0,904.247097,1012.773889,893.5234,1117.9624,13584.474886,17922.374429,15152.332374,0.0,...,15152.332374,0.0,0.0,255.479090,286.449450,295.2180,340.0821,13584.474886,17922.374429,15152.332374
2040-01-01 02:00:00,0.0,0.0,994.318173,1116.139825,837.1671,1080.4440,13584.474886,17922.374429,15152.332374,0.0,...,15152.332374,0.0,0.0,281.584037,316.476834,268.8070,320.3563,13584.474886,17922.374429,15152.332374
2040-01-01 03:00:00,0.0,0.0,1225.170282,1379.734844,802.7096,1070.9570,13584.474886,17922.374429,15152.332374,0.0,...,15152.332374,0.0,0.0,347.325056,391.735025,257.3263,310.2249,13584.474886,17922.374429,15152.332374
2040-01-01 04:00:00,0.0,0.0,1701.112830,1923.770538,775.9906,1085.4918,13584.474886,17922.374429,15152.332374,0.0,...,15152.332374,0.0,0.0,482.895680,547.112117,251.0431,312.4803,13584.474886,17922.374429,15152.332374


In [ ]:
print(f"Total number of columns: {len(demand_profiles_df.columns)}")
print("List of columns:")
print(demand_profiles_df.columns.tolist())

Total number of columns: 63
List of columns:
['ITSI_H2_zone_1', 'ITSI_H2_zone_2', 'ITSI_H2_heat', 'ITSI_CH4_heat', 'ITSI_El_market', 'ITSI_El_prosumer', 'ITSI_SNG', 'ITSI_eDiesel', 'ITSI_eKerosine', 'ITSA_H2_zone_1', 'ITSA_H2_zone_2', 'ITSA_H2_heat', 'ITSA_CH4_heat', 'ITSA_El_market', 'ITSA_El_prosumer', 'ITSA_SNG', 'ITSA_eDiesel', 'ITSA_eKerosine', 'ITS1_H2_zone_1', 'ITS1_H2_zone_2', 'ITS1_H2_heat', 'ITS1_CH4_heat', 'ITS1_El_market', 'ITS1_El_prosumer', 'ITS1_SNG', 'ITS1_eDiesel', 'ITS1_eKerosine', 'ITN1_H2_zone_1', 'ITN1_H2_zone_2', 'ITN1_H2_heat', 'ITN1_CH4_heat', 'ITN1_El_market', 'ITN1_El_prosumer', 'ITN1_SNG', 'ITN1_eDiesel', 'ITN1_eKerosine', 'ITCS_H2_zone_1', 'ITCS_H2_zone_2', 'ITCS_H2_heat', 'ITCS_CH4_heat', 'ITCS_El_market', 'ITCS_El_prosumer', 'ITCS_SNG', 'ITCS_eDiesel', 'ITCS_eKerosine', 'ITCN_H2_zone_1', 'ITCN_H2_zone_2', 'ITCN_H2_heat', 'ITCN_CH4_heat', 'ITCN_El_market', 'ITCN_El_prosumer', 'ITCN_SNG', 'ITCN_eDiesel', 'ITCN_eKerosine', 'ITCA_H2_zone_1', 'ITCA_H2_zone_2', 

In [ ]:
import os

# Define the output directory
output_dir = os.path.join(PROJECT_DIR, 'Italy')

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

# Construct the full file path
output_file = os.path.join(output_dir, f"{df_name}.csv")

# Save the dataframe to CSV
demand_profiles_df.to_csv(output_file)

print(f"Dataframe successfully saved to: {output_file}")


Dataframe successfully saved to: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/data/Italy/demand_profiles_GA_2040.csv


# Demand Data - NT

In [ ]:
year = 2030 # 2030 or 2040
scenario = "NT"

In [ ]:
def load_NT_hydrogen_demand(year, zones):
    """
    Loads the Hydrogen demand profile for the NT scenario and year.
    Handles specific subfolder structure and file names for the NT scenario.
    """
    print(f"--- Loading Hydrogen Demand (NT {year}) ---")
    demand_base_path = os.path.join(PROJECT_DIR, 'Demand_profiles')
    h2_file_path = os.path.join(demand_base_path, 'NT', 'H2 demand profiles', f'H2 {year}', f'NT_{year}.xlsx')

    h2_demand_df = load_zonal_profiles(h2_file_path, zones)
    return h2_demand_df

def load_NT_electricity_profiles(year, zones):
    """
    Loads Electricity Market and Prosumer profiles for the NT scenario.
    """
    print(f"--- Loading Electricity Profiles (NT {year}) ---")
    demand_base_path = os.path.join(PROJECT_DIR, 'Demand_profiles')

    elec_file_path = os.path.join(demand_base_path, 'NT', 'Electricity demand profiles', f'{year}_National Trends.xlsx')
    electricity_market_df = load_zonal_profiles(elec_file_path, zones)

    return electricity_market_df

# 1. Load Hydrogen Profiles
h2_demand = load_NT_hydrogen_demand(year, zones)

# 2. Load Electricity Profiles
elec_market_nt = load_NT_electricity_profiles(year, zones)

# Display sample results
print("\n--- Sample Data for NT (H2 Demand) ---")
display(h2_demand.head())

print("\n--- Sample Data for NT (Electricity Market) ---")
display(elec_market_nt.head())


--- Loading Hydrogen Demand (NT 2030) ---
Loading NT_2030.xlsx...
  -> AVAILABLE SHEETS IN FILE: ['AT00', 'BE00', 'BG00', 'CH00', 'CY00', 'CZ00', 'DE00', 'DKE1', 'EE00', 'ES00', 'FI00', 'FR00', 'GR00', 'HU00', 'HR00', 'IE00', 'ITN1', 'LU00', 'LT00', 'LV00', 'MT00', 'NL00', 'NO00', 'PL00', 'PT00', 'RO00', 'SK00', 'SI00', 'SE01', 'UK00']
  - Error loading zone ITSI: Worksheet named 'ITSI' not found. Filling with zeros.
  - Error loading zone ITSA: Worksheet named 'ITSA' not found. Filling with zeros.
  - Error loading zone ITS1: Worksheet named 'ITS1' not found. Filling with zeros.
  - Successfully loaded zone: ITN1
  - Error loading zone ITCS: Worksheet named 'ITCS' not found. Filling with zeros.
  - Error loading zone ITCN: Worksheet named 'ITCN' not found. Filling with zeros.
  - Error loading zone ITCA: Worksheet named 'ITCA' not found. Filling with zeros.
--- Loading Electricity Profiles (NT 2030) ---
Loading 2030_National Trends.xlsx...
  -> AVAILABLE SHEETS IN FILE: ['AL00', 'AT00

,ITSI,ITSA,ITS1,ITN1,ITCS,ITCN,ITCA
2040-01-01 00:00:00,0.0,0.0,0.0,2365.625173,0.0,0.0,0.0
2040-01-01 01:00:00,0.0,0.0,0.0,2365.705785,0.0,0.0,0.0
2040-01-01 02:00:00,0.0,0.0,0.0,2365.786398,0.0,0.0,0.0
2040-01-01 03:00:00,0.0,0.0,0.0,2365.867010,0.0,0.0,0.0
2040-01-01 04:00:00,0.0,0.0,0.0,2365.947623,0.0,0.0,0.0



--- Sample Data for NT (Electricity Market) ---


,ITSI,ITSA,ITS1,ITN1,ITCS,ITCN,ITCA
2040-01-01 00:00:00,1643.475755,1032.762368,1756.908334,9789.669401,3750.715446,1831.153442,516.273096
2040-01-01 01:00:00,1606.728939,1022.226336,1749.889472,9938.272161,3687.373070,1824.692632,522.001549
2040-01-01 02:00:00,1631.546285,1045.142540,1869.127502,11196.369487,3964.242569,1922.118265,576.890781
2040-01-01 03:00:00,1733.695496,1119.490514,2022.858280,13167.199550,4373.713461,2120.798384,632.942407
2040-01-01 04:00:00,1914.871091,1183.527452,2208.215819,14368.063892,4780.199858,2256.185900,709.346415
